# 🧹 Vacuum Search Challenge
**UNOESC · Inteligência Artificial · Equipe 404 AI**

## Planejamento no Vacuum World 5×5

Até aqui, construímos agentes que **reagem** ao ambiente e estudamos diferentes estratégias de busca.

Agora o objetivo é diferente:

> **Construir um agente planejador capaz de encontrar uma sequência de ações para limpar um ambiente 5×5.**

O agente deverá tratar o Vacuum World como um **problema de busca em espaço de estados**.

### Algoritmos permitidos

- BFS — Breadth-First Search
- DFS — Depth-First Search
- UCS — Uniform Cost Search
- A* — A-star

**Importante:** o objetivo não é apenas encontrar uma solução.

É necessário analisar **por que a estratégia escolhida é adequada ao problema**.

---
**Resumo da entrega (Equipe 404 AI):** Implementação do agente planejador `meu_planejador(initial_state)` utilizando o algoritmo **A\* (A-star)** guiado por uma **heurística admissível e consistente** composta pelo custo obrigatório de aspiração ($2 \times |D|$), **Árvore Geradora Mínima (MST — Prim)** sobre as salas sujas restantes com distância de Manhattan, distância até a sala suja mais próxima e **penalidade assimétrica de deslocamento para a esquerda** (`Left = 2`).

## 🎯 O desafio

Sua equipe deverá implementar uma função:

```python
def meu_planejador(initial_state):
    ...
```

Essa função deve receber o estado inicial e retornar um **plano de ações** que resolva o problema.

A solução será testada em diferentes mapas.

### Seu agente precisa

1. representar corretamente o estado;
2. gerar sucessores válidos;
3. identificar o objetivo;
4. encontrar um plano válido;
5. evitar ações desnecessárias;
6. considerar os custos do ambiente;
7. justificar tecnicamente a estratégia escolhida.

## 🗺️ Ambiente

O ambiente possui **5×5 posições**.

- `🤖` → posição inicial do agente
- `🦠` → sala suja
- `⬛` → obstáculo
- `✨` → sala limpa

O agente pode:

- mover para cima;
- mover para baixo;
- mover para esquerda;
- mover para direita;
- aspirar uma sala suja.

Uma ação de movimento só pode ser realizada quando a posição de destino estiver dentro do mapa e não for um obstáculo.

In [65]:
from dataclasses import dataclass
from heapq import heappush, heappop
from time import perf_counter

SIZE = 5

UP = "Up"
DOWN = "Down"
LEFT = "Left"
RIGHT = "Right"
SUCK = "Suck"

MOVES = {
    UP: (-1, 0),
    DOWN: (1, 0),
    LEFT: (0, -1),
    RIGHT: (0, 1),
}

ACTIONS = [UP, DOWN, LEFT, RIGHT, SUCK]

## 1. Representação do estado

Vamos representar um estado como:

```text
(posição do agente, salas sujas, obstáculos)
```

Exemplo:

```python
((2, 2), frozenset({(1, 1), (2, 3)}), frozenset({(1, 2), (3, 1))
```

Isso significa:

- agente na posição `(2, 2)`;
- salas `(1, 1)` e `(2, 3)` ainda estão sujas;
- salas `(1, 2)` e `(3, 1)` possuem obstáculos (não podem ser visitadas).

A representação do estado é fundamental porque os algoritmos de busca trabalham sobre **estados**, e não diretamente sobre o mapa visual.

In [66]:
State = tuple

def make_state(position, dirty_rooms, obstacles):
    return (
        position,
        frozenset(dirty_rooms),
        frozenset(obstacles)
    )

## 2. Mapa

O mapa contém obstáculos.

As ações `Suck` e `Left` possuem custo `2`.

As outras ações de movimento possuem custo `1`.

In [67]:
# Custos das ações
ACTION_COST = {
    "Up": 1,
    "Down": 1,
    "Left": 2,
    "Right": 1,
    "Suck": 2
}

In [68]:
def render_map(state):
    position, dirty_rooms, obstacles = state

    for r in range(SIZE):
        row = []
        for c in range(SIZE):
            cell = (r, c)

            if cell == position:
                symbol = "🤖"
            elif cell in obstacles:
                symbol = "⬛"
            elif cell in dirty_rooms:
                symbol = "🦠"
            else:
                symbol = "✨"

            row.append(symbol)

        print(" ".join(row))

## 3. Estado inicial de teste

Antes de desenvolver o agente, vamos testar a representação do problema em um cenário conhecido.

In [69]:
INITIAL_STATE = make_state(
    position=(2, 2),
    dirty_rooms={(0, 4), (2, 3), (4, 0), (4, 4)},
    obstacles={(1, 2), (3, 1)}
)

render_map(INITIAL_STATE)

✨ ✨ ✨ ✨ 🦠
✨ ✨ ⬛ ✨ ✨
✨ ✨ 🤖 🦠 ✨
✨ ⬛ ✨ ✨ ✨
🦠 ✨ ✨ ✨ 🦠


## 4. Função sucessora

A função sucessora recebe um estado e produz os estados que podem ser alcançados por uma ação válida.

Ela será fornecida pronta para que o foco da atividade esteja na **estratégia de busca e no planejamento**.

In [70]:
def successors(state):
    position, dirty_rooms, obstacles = state
    r, c = position

    for action, (dr, dc) in MOVES.items():
        new_position = (r + dr, c + dc)

        if not (0 <= new_position[0] < SIZE and 0 <= new_position[1] < SIZE):
            continue

        if new_position in obstacles:
            continue

        yield action, make_state(new_position, dirty_rooms, obstacles)

    if position in dirty_rooms:
        new_dirty = set(dirty_rooms)
        new_dirty.remove(position)
        yield SUCK, make_state(position, new_dirty, obstacles)

## 5. Teste do objetivo

O problema está resolvido quando **não existem mais salas sujas**.

In [71]:
def is_goal(state):
    #position, dirty_rooms, obstacles = state
    _, dirty_rooms, _ = state
    return len(dirty_rooms) == 0

## 6. Custo das ações

O custo acumulado será importante para comparar as estratégias.


In [72]:
def action_cost(action):
    return ACTION_COST[action]

def plan_cost(initial_state, plan):
    state = initial_state
    total = 0

    for action in plan:
        valid_actions = dict(successors(state))

        if action not in valid_actions:
            return None

        total += action_cost(action)
        state = valid_actions[action]

    return total if is_goal(state) else None


# 🛠️ 7. Implementação do planejador

Agora é a parte principal do Challenge.

Implemente:

```python
def meu_planejador(initial_state):
    ...
```

Você pode escolher:

- BFS
- DFS
- UCS
- A*

### Atenção

A escolha deve ser **justificada**.

Não basta dizer:

> “Escolhi A* porque é mais rápido.”

Explique:

- o que o algoritmo prioriza;
- como ele considera o custo;
- se utiliza heurística;
- quais propriedades são importantes para este problema;
- quais são suas limitações.

In [73]:
def meu_planejador(initial_state):
    """
    Agente planejador para o Vacuum World 5x5 utilizando busca informada A* (A-star).
    Equipe: 404 AI

    Avalia estados por f(n) = g(n) + h(n), onde:
      - g(n): custo real acumulado das ações desde o estado inicial;
      - h(n): heurística admissível e consistente que combina:
          1. Custo obrigatório de aspiração das salas sujas restantes (2 * |D|);
          2. Custo da Árvore Geradora Mínima (MST via algoritmo de Prim com
             distância de Manhattan) conectando as salas sujas restantes;
          3. Distância de Manhattan do agente até a sala suja mais próxima;
          4. Penalidade assimétrica obrigatória de movimento para a esquerda
             (+1 por coluna quando há sala suja à esquerda do agente, pois Left=2 e Right/Up/Down=1).
    """
    if is_goal(initial_state):
        meu_planejador.last_expanded = 0
        return []

    suck_cost = action_cost(SUCK)
    min_move_cost = min(action_cost(UP), action_cost(DOWN), action_cost(LEFT), action_cost(RIGHT))
    left_extra_cost = max(0, action_cost(LEFT) - action_cost(RIGHT))

    # Cache da MST e da coluna mais à esquerda por subconjunto de salas sujas (frozenset)
    mst_cache = {}

    def get_dirty_metrics(dirty_rooms):
        metrics = mst_cache.get(dirty_rooms)
        if metrics is not None:
            return metrics

        k = len(dirty_rooms)
        min_col = min(c for _, c in dirty_rooms)

        if k <= 1:
            res = (k * suck_cost, min_col)
            mst_cache[dirty_rooms] = res
            return res

        # Algoritmo de Prim para MST com distância de Manhattan entre salas sujas
        nodes = list(dirty_rooms)
        in_mst = [False] * k
        min_edge = [float("inf")] * k
        min_edge[0] = 0
        mst_dist = 0

        for _ in range(k):
            u = -1
            best = float("inf")
            for i in range(k):
                if not in_mst[i] and min_edge[i] < best:
                    best = min_edge[i]
                    u = i
            in_mst[u] = True
            mst_dist += best
            ur, uc = nodes[u]
            for v in range(k):
                if not in_mst[v]:
                    vr, vc = nodes[v]
                    dist = abs(ur - vr) + abs(uc - vc)
                    if dist < min_edge[v]:
                        min_edge[v] = dist

        res = (k * suck_cost + mst_dist * min_move_cost, min_col)
        mst_cache[dirty_rooms] = res
        return res

    def heuristic(state):
        pos, dirty_rooms, _ = state
        if not dirty_rooms:
            return 0

        base_cost, min_col = get_dirty_metrics(dirty_rooms)
        pr, pc = pos
        nearest_dist = min(abs(pr - dr) + abs(pc - dc) for dr, dc in dirty_rooms)
        left_penalty = (pc - min_col) * left_extra_cost if pc > min_col else 0

        return base_cost + nearest_dist * min_move_cost + left_penalty

    # Fila de prioridade (min-heap): (f, -g, tie_breaker, state, plan)
    # Desempate por -g prioriza estados mais profundos (mais próximos do objetivo) quando f empata
    h0 = heuristic(initial_state)
    frontier = [(h0, 0, 0, initial_state, [])]
    best_cost = {initial_state: 0}
    expanded = 0
    tie_breaker = 0

    while frontier:
        f_val, neg_g, _, state, plan = heappop(frontier)
        g_val = -neg_g

        if g_val > best_cost.get(state, float("inf")):
            continue

        if is_goal(state):
            meu_planejador.last_expanded = expanded
            return plan

        expanded += 1

        for action, next_state in successors(state):
            new_g = g_val + action_cost(action)
            if new_g < best_cost.get(next_state, float("inf")):
                best_cost[next_state] = new_g
                tie_breaker += 1
                new_f = new_g + heuristic(next_state)
                heappush(frontier, (new_f, -new_g, tie_breaker, next_state, plan + [action]))

    meu_planejador.last_expanded = expanded
    return None

## 9. Executando o planejador

Depois de implementar `meu_planejador`, execute:

In [74]:
start = perf_counter()

plan = meu_planejador(INITIAL_STATE)
elapsed = perf_counter() - start

print("Plano:", plan)

print("\nNúmero de ações:", len(plan) if plan is not None else None)
print("Custo total:", plan_cost(INITIAL_STATE, plan) if plan is not None else None)
print("Estados expandidos:", getattr(meu_planejador, "last_expanded", None))
print("Tempo:", f"{elapsed:.6f} s")

Plano: ['Right', 'Suck', 'Up', 'Up', 'Right', 'Suck', 'Down', 'Down', 'Down', 'Down', 'Suck', 'Left', 'Left', 'Left', 'Left', 'Suck']

Número de ações: 16
Custo total: 24
Estados expandidos: 20
Tempo: 0.001447 s


## 10. Validação do plano

Uma solução precisa ser **válida**, não apenas parecer correta.

A função abaixo executa o plano no ambiente e verifica se todas as salas foram limpas.

In [75]:
def execute_plan(initial_state, plan, verbose=False):
    state = initial_state
    total_cost = 0
    if verbose:
        print(f"\nEstado inicial:")
        render_map(state)

    if plan is None:
        return {
            "valid": False,
            "final_state": state,
            "cost": None,
            "actions": None,
        }

    for step, action in enumerate(plan, start=1):
        options = dict(successors(state))

        if action not in options:
            return {
                "valid": False,
                "final_state": state,
                "cost": total_cost,
                "actions": step - 1,
            }

        total_cost += action_cost(action)
        state = options[action]

        if verbose:
            print(f"\nPasso {step}: {action}")
            render_map(state)

    return {
        "valid": is_goal(state),
        "final_state": state,
        "cost": total_cost,
        "actions": len(plan),
    }

In [76]:

result = execute_plan(INITIAL_STATE, plan, verbose=True)

print("\nSolução válida:", result["valid"])
print("Custo:", result["cost"])
print("Ações:", result["actions"])



Estado inicial:
✨ ✨ ✨ ✨ 🦠
✨ ✨ ⬛ ✨ ✨
✨ ✨ 🤖 🦠 ✨
✨ ⬛ ✨ ✨ ✨
🦠 ✨ ✨ ✨ 🦠

Passo 1: Right
✨ ✨ ✨ ✨ 🦠
✨ ✨ ⬛ ✨ ✨
✨ ✨ ✨ 🤖 ✨
✨ ⬛ ✨ ✨ ✨
🦠 ✨ ✨ ✨ 🦠

Passo 2: Suck
✨ ✨ ✨ ✨ 🦠
✨ ✨ ⬛ ✨ ✨
✨ ✨ ✨ 🤖 ✨
✨ ⬛ ✨ ✨ ✨
🦠 ✨ ✨ ✨ 🦠

Passo 3: Up
✨ ✨ ✨ ✨ 🦠
✨ ✨ ⬛ 🤖 ✨
✨ ✨ ✨ ✨ ✨
✨ ⬛ ✨ ✨ ✨
🦠 ✨ ✨ ✨ 🦠

Passo 4: Up
✨ ✨ ✨ 🤖 🦠
✨ ✨ ⬛ ✨ ✨
✨ ✨ ✨ ✨ ✨
✨ ⬛ ✨ ✨ ✨
🦠 ✨ ✨ ✨ 🦠

Passo 5: Right
✨ ✨ ✨ ✨ 🤖
✨ ✨ ⬛ ✨ ✨
✨ ✨ ✨ ✨ ✨
✨ ⬛ ✨ ✨ ✨
🦠 ✨ ✨ ✨ 🦠

Passo 6: Suck
✨ ✨ ✨ ✨ 🤖
✨ ✨ ⬛ ✨ ✨
✨ ✨ ✨ ✨ ✨
✨ ⬛ ✨ ✨ ✨
🦠 ✨ ✨ ✨ 🦠

Passo 7: Down
✨ ✨ ✨ ✨ ✨
✨ ✨ ⬛ ✨ 🤖
✨ ✨ ✨ ✨ ✨
✨ ⬛ ✨ ✨ ✨
🦠 ✨ ✨ ✨ 🦠

Passo 8: Down
✨ ✨ ✨ ✨ ✨
✨ ✨ ⬛ ✨ ✨
✨ ✨ ✨ ✨ 🤖
✨ ⬛ ✨ ✨ ✨
🦠 ✨ ✨ ✨ 🦠

Passo 9: Down
✨ ✨ ✨ ✨ ✨
✨ ✨ ⬛ ✨ ✨
✨ ✨ ✨ ✨ ✨
✨ ⬛ ✨ ✨ 🤖
🦠 ✨ ✨ ✨ 🦠

Passo 10: Down
✨ ✨ ✨ ✨ ✨
✨ ✨ ⬛ ✨ ✨
✨ ✨ ✨ ✨ ✨
✨ ⬛ ✨ ✨ ✨
🦠 ✨ ✨ ✨ 🤖

Passo 11: Suck
✨ ✨ ✨ ✨ ✨
✨ ✨ ⬛ ✨ ✨
✨ ✨ ✨ ✨ ✨
✨ ⬛ ✨ ✨ ✨
🦠 ✨ ✨ ✨ 🤖

Passo 12: Left
✨ ✨ ✨ ✨ ✨
✨ ✨ ⬛ ✨ ✨
✨ ✨ ✨ ✨ ✨
✨ ⬛ ✨ ✨ ✨
🦠 ✨ ✨ 🤖 ✨

Passo 13: Left
✨ ✨ ✨ ✨ ✨
✨ ✨ ⬛ ✨ ✨
✨ ✨ ✨ ✨ ✨
✨ ⬛ ✨ ✨ ✨
🦠 ✨ 🤖 ✨ ✨

Passo 14: Left
✨ ✨ ✨ ✨ ✨
✨ ✨ ⬛ ✨ ✨
✨ ✨ ✨ ✨ ✨
✨ ⬛ ✨ ✨ ✨
🦠 🤖 ✨ ✨ ✨

Passo 15: Left
✨ ✨ 

# 📊 11. Comparando estratégias

Agora que o agente funciona, não basta perguntar:

> “Funcionou?”

Também queremos saber:

- encontrou solução?
- qual foi o custo?
- quantas ações foram realizadas?
- quanto tempo levou?
- quantos estados foram expandidos?

Uma solução válida pode não ser uma solução eficiente.

In [77]:
def evaluate_plan(initial_state, plan, expanded=None):
    start = perf_counter()
    result = execute_plan(initial_state, plan)
    elapsed = perf_counter() - start

    return {
        "valid": result["valid"],
        "actions": result["actions"],
        "cost": result["cost"],
        "expanded": expanded,
        "time": elapsed,
    }

# Implementações de referência das demais estratégias para comparação experimental
from collections import deque

def planejador_bfs(initial_state):
    if is_goal(initial_state):
        return [], 0
    queue = deque([(initial_state, [])])
    visited = {initial_state}
    expanded = 0
    while queue:
        state, p = queue.popleft()
        expanded += 1
        for action, next_state in successors(state):
            if next_state not in visited:
                new_plan = p + [action]
                if is_goal(next_state):
                    return new_plan, expanded
                visited.add(next_state)
                queue.append((next_state, new_plan))
    return None, expanded

def planejador_dfs(initial_state):
    if is_goal(initial_state):
        return [], 0
    stack = [(initial_state, [])]
    visited = {initial_state}
    expanded = 0
    while stack:
        state, p = stack.pop()
        expanded += 1
        for action, next_state in successors(state):
            if next_state not in visited:
                new_plan = p + [action]
                if is_goal(next_state):
                    return new_plan, expanded
                visited.add(next_state)
                stack.append((next_state, new_plan))
    return None, expanded

def planejador_ucs(initial_state):
    if is_goal(initial_state):
        return [], 0
    frontier = [(0, 0, initial_state, [])]
    best_cost = {initial_state: 0}
    expanded = 0
    tie = 0
    while frontier:
        cost, _, state, p = heappop(frontier)
        if cost > best_cost.get(state, float("inf")):
            continue
        if is_goal(state):
            return p, expanded
        expanded += 1
        for action, next_state in successors(state):
            new_cost = cost + action_cost(action)
            if new_cost < best_cost.get(next_state, float("inf")):
                best_cost[next_state] = new_cost
                tie += 1
                heappush(frontier, (new_cost, tie, next_state, p + [action]))
    return None, expanded

def planejador_astar_wrapper(initial_state):
    p = meu_planejador(initial_state)
    return p, getattr(meu_planejador, "last_expanded", None)

# Cenários de benchmark (incluindo cenário que evidencia a falha de otimalidade do BFS quando Left=2)
BENCHMARK_SCENARIOS = [
    ("Cenário Inicial (Teste)", INITIAL_STATE),
    ("Mapa 1 (7 salas sujas)", make_state((0, 0), {(0, 4), (2, 2), (4, 0), (4, 1), (4, 2), (4, 3), (4, 4)}, {(1, 2), (3, 1), (2, 4), (3, 4)})),
    ("Mapa 2 (Já limpo)", make_state((4, 0), {}, {(1, 2), (3, 1), (2, 4), (3, 4)})),
    ("Mapa 3 (3 cantos + obstáculos)", make_state((2, 2), {(0, 4), (4, 0), (4, 4)}, {(1, 2), (3, 1), (2, 4), (3, 4)})),
    ("Mapa Extra (Assimetria Left=2)", make_state((1, 2), {(0, 0), (1, 4)}, {(2, 2)})),
]

ALGORITHMS = [
    ("BFS", planejador_bfs),
    ("DFS", planejador_dfs),
    ("UCS", planejador_ucs),
    ("A* (404 AI)", planejador_astar_wrapper),
]

print(f"{'Cenário':<32} | {'Algoritmo':<12} | {'Válido':<6} | {'Ações':>5} | {'Custo':>5} | {'Expandidos':>10} | {'Tempo (ms)':>10}")
print("-" * 96)

for scen_name, scen_state in BENCHMARK_SCENARIOS:
    for alg_name, alg_fn in ALGORITHMS:
        t0 = perf_counter()
        cand_plan, exp_nodes = alg_fn(scen_state)
        search_time_ms = (perf_counter() - t0) * 1000
        metrics = evaluate_plan(scen_state, cand_plan, expanded=exp_nodes)
        print(
            f"{scen_name:<32} | {alg_name:<12} | {str(metrics['valid']):<6} | "
            f"{metrics['actions']:>5} | {metrics['cost']:>5} | {metrics['expanded']:>10} | {search_time_ms:>10.3f}"
        )
    print("-" * 96)

Cenário                          | Algoritmo    | Válido | Ações | Custo | Expandidos | Tempo (ms)
------------------------------------------------------------------------------------------------
Cenário Inicial (Teste)          | BFS          | True   |    16 |    24 |        271 |      3.622
Cenário Inicial (Teste)          | DFS          | True   |    28 |    40 |         28 |      0.272
Cenário Inicial (Teste)          | UCS          | True   |    16 |    24 |        311 |      5.999
Cenário Inicial (Teste)          | A* (404 AI)  | True   |    16 |    24 |         20 |      0.926
------------------------------------------------------------------------------------------------
Mapa 1 (7 salas sujas)           | BFS          | True   |    23 |    32 |       2428 |     56.381
Mapa 1 (7 salas sujas)           | DFS          | True   |    34 |    50 |         35 |      0.765
Mapa 1 (7 salas sujas)           | UCS          | True   |    23 |    32 |       2508 |     48.900
Mapa 1 (7 sala

# 🔬 12. Testando em diferentes mapas

Uma estratégia que funciona em um único cenário pode apresentar outro comportamento em mapas diferentes.

Teste seu planejador em vários cenários.

A função abaixo permite criar novos casos sem alterar o algoritmo.

In [78]:
TEST_CASES = [
    make_state(
        (0, 0),
        {(0, 4), (2, 2), (4, 0), (4, 1), (4, 2), (4, 3), (4, 4)},
        {(1,2), (3,1), (2,4), (3,4)}
    ),

    make_state(
        (4, 0),
        {},
        {(1,2), (3,1), (2,4), (3,4)}
    ),

    make_state(
        (2, 2),
        {(0, 4), (4, 0), (4, 4)},
        {(1,2), (3,1), (2,4), (3,4)}
    ),
]

In [79]:
for i, test_state in enumerate(TEST_CASES, start=1):
    print("=" * 60)
    print(f"MAPA {i}")
    print("=" * 60)

    render_map(test_state)

    start = perf_counter()
    test_plan = meu_planejador(test_state)
    elapsed = perf_counter() - start

    result = execute_plan(test_state, test_plan)

    print("\nPlano encontrado:", test_plan)
    print("Válido:", result["valid"])
    print("Ações:", result["actions"])
    print("Custo:", result["cost"])
    print("Estados expandidos:", getattr(meu_planejador, "last_expanded", None))
    print("Tempo:", f"{elapsed:.6f} s")

MAPA 1
🤖 ✨ ✨ ✨ 🦠
✨ ✨ ⬛ ✨ ✨
✨ ✨ 🦠 ✨ ⬛
✨ ⬛ ✨ ✨ ⬛
🦠 🦠 🦠 🦠 🦠

Plano encontrado: ['Down', 'Down', 'Down', 'Down', 'Suck', 'Right', 'Suck', 'Right', 'Suck', 'Right', 'Suck', 'Right', 'Suck', 'Left', 'Left', 'Up', 'Up', 'Suck', 'Right', 'Up', 'Up', 'Right', 'Suck']
Válido: True
Ações: 23
Custo: 32
Estados expandidos: 73
Tempo: 0.003283 s
MAPA 2
✨ ✨ ✨ ✨ ✨
✨ ✨ ⬛ ✨ ✨
✨ ✨ ✨ ✨ ⬛
✨ ⬛ ✨ ✨ ⬛
🤖 ✨ ✨ ✨ ✨

Plano encontrado: []
Válido: True
Ações: 0
Custo: 0
Estados expandidos: 0
Tempo: 0.000007 s
MAPA 3
✨ ✨ ✨ ✨ 🦠
✨ ✨ ⬛ ✨ ✨
✨ ✨ 🤖 ✨ ⬛
✨ ⬛ ✨ ✨ ⬛
🦠 ✨ ✨ ✨ 🦠

Plano encontrado: ['Left', 'Left', 'Down', 'Down', 'Suck', 'Right', 'Right', 'Right', 'Right', 'Suck', 'Left', 'Up', 'Up', 'Up', 'Up', 'Right', 'Suck']
Válido: True
Ações: 17
Custo: 23
Estados expandidos: 36
Tempo: 0.001054 s


# 🏆 13. Vacuum Search Challenge

## 📦 Entrega da equipe — **404 AI**

### Código

```python
def meu_planejador(initial_state):
    # Implementado na Seção 7 utilizando o algoritmo A* (A-star)
    # com heurística admissível e consistente (MST + Distância de Manhattan + Penalidade Assimétrica Left).
    ...
```

---

### Estratégia escolhida — Respostas Técnicas (Equipe 404 AI)

#### **1. Qual algoritmo sua equipe utilizou?**
Utilizamos o algoritmo de busca informada **A\* (*A-star*)**, implementado com fila de prioridade (*min-heap* via `heapq`), controle de estados visitados com melhor custo conhecido (`best_cost`), memoização da Árvore Geradora Mínima (*MST*) por subconjunto de salas sujas e critério de desempate por maior custo acumulado $g(n)$ (priorizando nós mais proundos e próximos do estado objetivo em caso de empate em $f(n)$).

---

#### **2. Por que essa estratégia é adequada ao problema?**
O *Vacuum World 5×5* com múltiplas salas sujas, obstáculos e **custos de ação não uniformes** (`Up = 1`, `Down = 1`, `Right = 1`, `Left = 2`, `Suck = 2`) é uma variante do Problema do Caixeiro Viajante em grade (*Grid Pathfinding + Travelling Salesperson Problem*) sobre um espaço de estados de tamanho até $25 \times 2^{|D|}$. Analisando as quatro estratégias estudadas:

- **DFS (*Depth-First Search*):** Não é ótima nem em número de passos nem em custo. Como explora cegamente um ramo em profundidade antes de retroceder, gera planos longos com movimentos redundantes (por exemplo, custo **40** com 28 ações no estado inicial de teste e custo **50** com 34 ações no Mapa 1, contra os custos ótimos **24** e **32**).
- **BFS (*Breadth-First Search*):** Minimiza apenas a **quantidade de ações** (profundidade $d$), ignorando completamente o dicionário `ACTION_COST`. Como `Left` custa **2** e os demais movimentos custam **1**, um plano com *menos ações* pode ter *custo maior*! Por exemplo, no **Mapa Extra (`pos=(1,2)`, sujas `{(0,0), (1,4)}`, obstáculo `{(2,2)}`)**, o BFS escolhe ir primeiro para `(1,4)` e depois para `(0,0)` em **9 ações**, pagando 4 movimentos `Left` e resultando em **custo 15**; já o plano ótimo vai primeiro para `(0,0)` e depois para `(1,4)` em **10 ações** (2 `Left` e 4 `Right`), com **custo 14**. Além disso, o BFS expande muitos estados desnecessários (**2.428 estados** no Mapa 1).
- **UCS (*Uniform Cost Search*):** Garante encontrar o plano de **custo mínimo**, pois expande os estados em ordem crescente do custo acumulado $g(n)$. Contudo, por ser uma busca cega (sem informação sobre a localização das sujeiras restantes), expande estados em todas as direções possíveis (**311 estados** no cenário inicial e **2.508 estados** no Mapa 1).
- **A\* (*A-star* — Escolha da Equipe):** Reúne a **otimalidade garantida** do UCS com a **eficiência direcionada** pela função heurística $h(n)$. Nos experimentos realizados:
  - No **Cenário Inicial**, encontrou o plano ótimo (**custo 24**, 16 ações) expandindo apenas **20 estados** (**93,6% menos expansões** que o UCS com 311 e **92,6% menos** que o BFS com 271).
  - No **Mapa 1** (7 salas sujas), encontrou o plano ótimo (**custo 32**, 23 ações) expandindo apenas **73 estados** (**97,1% menos expansões** que o UCS com 2.508 e **97,0% menos** que o BFS com 2.428).
  - No **Mapa 3**, encontrou o plano ótimo (**custo 23**, 17 ações) expandindo apenas **36 estados** (contra 134 do UCS e 126 do BFS).

---

#### **3. O algoritmo considera custo? Como?**
**Sim.** O algoritmo A\* considera o custo tanto de forma **retrospectiva (custo real acumulado)** quanto **prospectiva (custo futuro estimado)** por meio da função de avaliação:

$$f(n) = g(n) + h(n)$$

1. **Custo acumulado real $g(n)$:** A cada transição $(s, a, s')$ gerada por `successors(state)`, o algoritmo consulta `action_cost(action)` (`Up: 1`, `Down: 1`, `Right: 1`, `Left: 2`, `Suck: 2`) e acumula:
   $$g(s') = g(s) + \text{action\_cost}(a)$$
   O dicionário `best_cost[state]` registra o menor custo $g$ já encontrado para cada estado. Se um mesmo estado $(pos, D, O)$ for alcançado por um caminho mais barato ($g_{\text{novo}} < \text{best\_cost}[s']$), o estado é atualizado e reinserido na fila de prioridade.
2. **Custo futuro estimado $h(n)$:** A heurística incorpora diretamente os custos reais de `Suck` (`2` por sala suja), o custo mínimo de deslocamento por célula (`1`) e o custo adicional obrigatório de deslocamento para a esquerda (`Left - Right = 2 - 1 = +1` por coluna à esquerda da posição atual).
3. **Priorização na fronteira:** A fila de prioridade (`heapq`) extrai sempre o nó com menor $f(n) = g(n) + h(n)$, penalizando trajetórias que abusam da ação `Left` (custo 2) ou que fazem desvios desnecessários.

---

#### **4. Se utilizou A\*, qual é a heurística utilizada?**
Para um estado $s = (\text{pos}, D, O)$, onde $\text{pos} = (r_p, c_p)$ é a posição atual do robô e $D$ é o conjunto de salas sujas restantes ($|D| = k$), se $D = \emptyset$ então $h(s) = 0$. Caso contrário ($k \ge 1$), definimos:

$$h(s) = \underbrace{2|D|}_{\text{custo de Suck}} + \underbrace{\text{MST}_{\text{Manhattan}}(D)}_{\text{conexão entre salas sujas}} + \underbrace{\min_{(r_d, c_d) \in D} \left(|r_p - r_d| + |c_p - c_d|\right)}_{\text{distância até a sujeira mais próxima}} + \underbrace{\max\!\left(0,\; c_p - \min_{(r_d, c_d) \in D} c_d\right)}_{\text{penalidade assimétrica de Left}}$$

**Componentes da heurística e prova de admissibilidade ($0 \le h(s) \le h^*(s)$):**
1. **Custo obrigatório de limpeza ($2|D|$):** Cada sala suja em $D$ exige pelo menos uma ação `Suck` (custo 2) exclusiva naquela célula. Logo, qualquer plano válido gasta exatamente $2|D|$ em aspirações.
2. **Distância de Manhattan até a sala suja mais próxima ($\min_{d \in D} d_{\text{Manh}}(\text{pos}, d)$):** Antes de limpar a primeira sala de $D$, o robô precisa se deslocar de $\text{pos}$ até alguma sala $d_1 \in D$, o que exige no mínimo $|r_p - r_{d_1}| + |c_p - c_{d_1}| \ge \min_{d \in D} d_{\text{Manh}}(\text{pos}, d)$ movimentos (mesmo sem obstáculos).
3. **Árvore Geradora Mínima sobre $D$ ($\text{MST}_{\text{Manhattan}}(D)$):** Para visitar todas as salas em $D = \{d_1, \dots, d_k\}$, o caminho percorrido pelo robô entre as salas forma uma árvore geradora sobre os vértices de $D$. Como o peso de qualquer caminho que conecta todos os pontos de $D$ é maior ou igual ao peso da Árvore Geradora Mínima (calculada via algoritmo de Prim com distância de Manhattan e armazenada em cache por `frozenset`), $\text{MST}_{\text{Manhattan}}(D)$ é um limite inferior estrito do número de movimentos necessários após alcançar a primeira sala suja.
4. **Penalidade assimétrica de deslocamento à esquerda ($\max(0, c_p - c_{\min})$):** Cada movimento horizontal, vertical ou `Right` custa pelo menos `1` por unidade de distância de Manhattan, enquanto cada ação `Left` custa `2 = 1 + 1` (ou seja, `+1` de sobretaxa além da distância de Manhattan). Para alcançar a sala suja mais à esquerda, situada na coluna $c_{\min} = \min_{(r_d, c_d) \in D} c_d$, qualquer trajetória que parta da coluna $c_p$ terá que executar a ação `Left` no mínimo $\max(0, c_p - c_{\min})$ vezes. Somar essa sobretaxa preserva rigorosamente a admissibilidade e a consistência, além de orientar o A\* a planejar rotas no sentido esquerda $\to$ direita quando vantajoso.

**Propriedades garantidas e limitações:**
- **Completude:** Garantida em qualquer grade finita (retorna o plano ou `None` se uma sala suja estiver isolada por obstáculos).
- **Otimalidade e Consistência:** Como $h(s) \le c(s, a, s') + h(s')$ para toda transição válida e $h(\text{goal}) = 0$, o A\* é **100% ótimo em custo**.
- **Limitações:** Se o número de salas sujas $|D|$ crescer muito em grades grandes, o espaço de estados cresce exponencialmente ($O(|V| \cdot 2^{|D|})$) e o uso de memória na fronteira aumenta — embora para o ambiente $5\times 5$ a combinação de `MST` com cache e penalidade assimétrica reduza as expansões para poucas dezenas de estados e tempo de execução na casa de $\sim 0{,}3\text{ a }2\text{ ms}$.

Foi utilizado IA como auxiliadora para resolução e complementação dos textos de explicação, apenas para melhorar o aprofundamento das questões e da atividade proposta!

In [80]:
print("Resumo Final — Equipe 404 AI | Estratégia: A* (A-star) com Heurística MST + Penalidade Assimétrica")
print("=" * 92)
all_cases = [("INITIAL_STATE", INITIAL_STATE)] + [(f"MAPA {i}", tc) for i, tc in enumerate(TEST_CASES, start=1)]
for label, st in all_cases:
    t0 = perf_counter()
    pl = meu_planejador(st)
    dt = (perf_counter() - t0) * 1000
    res = execute_plan(st, pl)
    exp = getattr(meu_planejador, "last_expanded", 0)
    print(f"{label:<15} | Válido: {str(res['valid']):<5} | Ações: {res['actions']:>2} | Custo: {res['cost']:>2} | Expandidos: {exp:>3} | Tempo: {dt:.3f} ms")

Resumo Final — Equipe 404 AI | Estratégia: A* (A-star) com Heurística MST + Penalidade Assimétrica
INITIAL_STATE   | Válido: True  | Ações: 16 | Custo: 24 | Expandidos:  20 | Tempo: 0.889 ms
MAPA 1          | Válido: True  | Ações: 23 | Custo: 32 | Expandidos:  73 | Tempo: 3.120 ms
MAPA 2          | Válido: True  | Ações:  0 | Custo:  0 | Expandidos:   0 | Tempo: 0.007 ms
MAPA 3          | Válido: True  | Ações: 17 | Custo: 23 | Expandidos:  36 | Tempo: 1.815 ms
